# Eval-awareness probes on a Colab A100, driven from VS Code

**Setup.** Install the official `Google.colab` extension, open this notebook, then
*Select Kernel -> Colab -> A100 GPU* and sign in. Pick the **High-RAM** shape: the
model is converted on the CPU before it moves to the GPU (see step 6).

## Read this before running anything

The kernel is **remote**. This `.ipynb` is edited locally, but every cell executes on a
Colab VM with its own filesystem, and **the extension does not sync your workspace**.
So even though `src/`, `scripts/`, and `datasets/` sit right next to this file on disk,
the runtime cannot see them -- `sys.path.append('.')` inside the scripts will not find
the repo. Getting the code onto the VM (step 2) is a real step, not a formality.

Two Colab conveniences are also unavailable here and are deliberately absent below:
`files.upload()` and `drive.mount()` both depend on the Colab web frontend's JS bridge
and are reported broken under the VS Code extension
([#250](https://github.com/googlecolab/colab-vscode/issues/250),
[#264](https://github.com/googlecolab/colab-vscode/issues/264)). `notebook_login()` is
replaced with a `getpass` prompt for the same reason. Anything written by these cells
lives on the VM and dies with the session -- see step 8 to get results back.

## Model: why Phi-3-medium-**4k**, not 128k

The README's Llama-3.3-70B is ~140 GB in bf16 and TransformerLens has no quantized load
path, so it is out of reach at any Colab tier -- Colab gives you one GPU, and the A100 is
the 40 GB card. This notebook runs `microsoft/Phi-3-medium-4k-instruct` (14B, 40 layers,
d_model 5120, ~28 GB in bf16) instead.

The **128k** variant of the same weights cannot work with this repo, for two independent
reasons, so do not substitute it back in:

1. It declares `max_position_embeddings=131072`. TransformerLens materialises a
   `[n_ctx, n_ctx]` boolean causal mask *per attention layer*
   (`abstract_attention.py`: `torch.tril(torch.ones((n_ctx, n_ctx)).bool())`), which is
   ~17 GB per layer at that context -- an OOM before a single token is processed. The 4k
   variant needs 16 MB per layer.
2. It sets `rope_scaling: "su"` with short_factor ramping 1.0 -> 16.8. TransformerLens's
   `Phi3ForCausalLM` config branch ignores `rope_scaling` entirely, so the activations
   would be silently wrong even if the mask fit.

Neither medium variant is in TransformerLens's `OFFICIAL_MODEL_NAMES`, so
`from_pretrained` rejects the name outright. `src/utils.py` registers the 4k one at
runtime (`register_unofficial_models`); the config branch derives shape from the HF
config and `convert_phi3_weights` already handles Phi-3-medium's grouped-query attention
(40 query heads / 10 KV heads), so nothing else needs patching.

If the load fails anyway, fall back to `microsoft/Phi-3-mini-4k-instruct` -- officially
supported, no patch, but 3.8B and a correspondingly weaker test.

## 1. Confirm we're actually on a remote A100

If this reports no GPU, the kernel silently fell back to a local Python interpreter --
re-pick the kernel. `Total RAM` under ~50 GB means you did not get the High-RAM shape.

In [ ]:
import subprocess, sys, os
print('python :', sys.executable)
print('cwd    :', os.getcwd())
print('colab  :', os.path.exists('/content'))
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv'], capture_output=True, text=True).stdout)
with open('/proc/meminfo') as f:
    print('Total RAM:', round(int(f.readline().split()[1]) / 1e6, 1), 'GB')

## 2. Get the repo onto the runtime

**Option A -- git clone (recommended).** Reliable and repeatable. The proto repo is
private, so the cell prompts for a fine-grained PAT with read access to Contents; a
tokenless clone hangs on git's credential prompt and dies rather than 404ing. Making
the repo public removes the prompt entirely -- leave the token blank if you do.

**Option B -- upload a zip** via the extension's Colab *Files* pane, drop it in
`/content`, then unzip below. No GitHub needed, but it's manual and has to be redone
every session.

In [ ]:
# Option A -- clone. The repo is private, so an unauthenticated clone does not fail
# loudly: git falls through to a username prompt, finds no TTY on the VM, and dies with
# "could not read Username for 'https://github.com'". Supply a PAT (GitHub -> Settings ->
# Developer settings -> Personal access tokens -> Fine-grained, Contents: read-only).
# subprocess, not `!`, so the token never lands in shell history or cell output.
import subprocess, shutil
from getpass import getpass

OWNER, REPO_NAME = 'ianshxn', 'eval-awareness-probe-proto'
TOKEN = getpass('GitHub PAT (leave blank if the repo is public): ').strip()

shutil.rmtree('/content/eap', ignore_errors=True)   # make re-runs idempotent
url = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{OWNER}/{REPO_NAME}.git"
r = subprocess.run(['git', 'clone', '-q', url, '/content/eap'],
                   capture_output=True, text=True)
if r.returncode:
    raise SystemExit(r.stderr.replace(TOKEN, '***') if TOKEN else r.stderr)
print('cloned ok')

# Option B: upload eval-awareness-probe-proto.zip via the Files pane, then:
# !mkdir -p /content/eap && unzip -q -o /content/eval-awareness-probe-proto.zip -d /content/eap

%cd /content/eap
!ls

## 3. Dependencies

`transformers==4.49.0` is pinned for TransformerLens compatibility and downgrades the
preinstalled version, so the kernel must restart (notebook toolbar -> **Restart**).

The VM's disk survives that restart -- the clone from step 2 is still there -- but the
Python process doesn't, so `%cd` and all variables reset. After restarting, re-run
step 4 onward; it re-establishes the working directory on its own.

In [ ]:
!pip install -q -r requirements.txt
print('Dependencies installed. Restart the kernel now, then continue at step 4.')

## 4. Re-establish state after the restart

In [ ]:
%cd /content/eap
MODEL = 'microsoft/Phi-3-medium-4k-instruct'
# Fallback if the medium load fails: 'microsoft/Phi-3-mini-4k-instruct'

import transformers, torch
print('transformers', transformers.__version__, '(expect 4.49.0)')
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0))

## 5. Hugging Face auth

Phi-3 is MIT-licensed and not gated, so this is only needed if you hit a rate limit or
swap in a gated model such as Llama. `getpass` prompts through VS Code's command palette
and the token goes straight to the remote kernel without being written into the notebook.

In [ ]:
from getpass import getpass
from huggingface_hub import login
login(token=getpass('HF read token: '))

## 6. Generate vectors (README step 3)

204 contrastive examples x 2 forward passes. The activation hook prints per layer per
example, so output is very chatty -- VS Code truncates long streams, which is fine since
the vectors go to disk. `--layers` is capped for a first run; drop it to sweep all 40.

**Expect the load to be slow and RAM-hungry.** TransformerLens pulls the HF model onto
the CPU, builds a converted state dict, and only then moves to the GPU, so peak host RAM
lands well above the 28 GB of weights -- this is the step that needs High-RAM. On the GPU
side, 28 GB of weights plus ~0.7 GB of causal masks leaves roughly 10 GB of headroom on a
40 GB A100.

In [ ]:
!python scripts/generate_vectors.py \
  --model $MODEL \
  --data datasets/contrastive_dataset.json \
  --output phi3medium_contrastive \
  --layers 10,15,20,25,30,35

In [ ]:
import glob
run_dir = sorted(glob.glob('phi3medium_contrastive_*'))[-1]
print(run_dir)
!cat {run_dir}/experiment_info.txt
!ls {run_dir}/normalized_vectors

## 7. Build a labelled test set, then analyze (README step 4)

`analyze_probe.py` labels each example with `item.get('type') == 'testing'`, but none of
the four shipped datasets carry a `type` field -- every example falls through to
`'deployment'`, the ROC gets a single class, and the AUC is undefined. Run the README's
step 4 as written and the plots render but mean nothing.

`scripts/build_labelled_eval.py` fixes this from data already in the repo:
`tom_vs_sharegpt.json` holds theory-of-mind benchmark items (testing) paired against real
ShareGPT user turns (deployment), packed two-to-a-row as `(A) ... \n(B) ...`. Splitting
each row in two yields 1076 labelled single-prompt examples, and labelling by content
rather than slot removes the A/B position confound.

**Use `--length-match`.** The raw split has a severe length confound: ToM items run
~5x longer than ShareGPT turns (median 570 vs 104 chars), and `probe_analysis` scores each
example by its *mean* projection across token positions. Prompt length alone separates
the classes at AUROC 0.82, so an unmatched run tells you almost nothing about the probe.
Length matching drops that baseline to 0.51 -- chance -- at the cost of keeping 147 pairs.
Anything the probe scores above 0.51 there is real signal.

In [ ]:
!python scripts/build_labelled_eval.py \
  --data datasets/tom_vs_sharegpt.json \
  --output datasets/eval_vs_deploy_labelled.json \
  --length-match

`analyze_dataset_with_probe` runs a **fresh forward pass per example per layer**, so the
294 length-matched examples x 6 layers is 1764 forward passes through a 14B model. Start
with `--limit` on the builder above if you want a faster first pass.

In [ ]:
!python scripts/analyze_probe.py \
  --model $MODEL \
  --vectors {run_dir}/normalized_vectors \
  --data datasets/eval_vs_deploy_labelled.json \
  --output {run_dir}/analysis \
  --visualize

`process_directory` prints AUROC per layer above. Note the polarity convention in
`calculate_auroc_from_csv`: `y_true` is 1 for **deployment**, so a probe that points at
the testing direction shows up as AUROC well *below* 0.5 rather than above it -- read
`abs(auroc - 0.5)` as the strength and the side as the sign.

Add `--control` to the cell above to compare against random probes with matched
statistics. That is the check worth running before believing any single number.

In [ ]:
# Plots render inline over the kernel protocol, so these do come back to VS Code.
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f'{run_dir}/analysis/visualizations/*.png')):
    print(p); display(Image(p))

## 8. Get results off the VM

The remote kernel cannot write to your local disk, and `drive.mount()` is unreliable
here -- so nothing above is saved anywhere durable yet.

The vectors are small (5120 floats per layer), so pushing them to a private HF dataset
repo is the least painful programmatic route. Manual alternative: zip the run directory
and pull it down through the extension's Colab *Files* pane.

In [ ]:
!zip -qr {run_dir}.zip {run_dir}
!ls -lh {run_dir}.zip   # then download via the Files pane

# Or push to a private HF repo:
# from huggingface_hub import HfApi
# api = HfApi()
# api.create_repo('YOUR_USER/eap-vectors', repo_type='dataset',
#                 private=True, exist_ok=True)
# api.upload_folder(folder_path=run_dir, repo_id='YOUR_USER/eap-vectors',
#                   repo_type='dataset')